### Task II: Classical Graph Neural Network (GNN)
For Task II, you will use ParticleNet’s data for Quark/Gluon jet classification available here with its corresponding description.
- Choose 2 Graph-based architectures of your choice to classify jets as being quarks or gluons. Provide a description on what considerations you have taken to project this point-cloud dataset to a set of interconnected nodes and edges.
- Discuss the resulting performance of the 2 chosen architectures.


In [1]:
!pip install torch_geometric

In [2]:
import warnings
warnings.filterwarnings('ignore')

In [3]:
# ==============================
# Quark/Gluon Jet Classification with GCN & GAT
# Using 1000 samples from .npz dataset
# ==============================

import numpy as np
import torch
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, GATConv
from torch_geometric.data import Data, DataLoader
from sklearn.model_selection import train_test_split

# ------------------------------
# Step 1: Load dataset (.npz)
# ------------------------------
dataset_path = "/content/QG_jets.npz"  # replace with actual filename
data = np.load(dataset_path)

jets = data["X"]        # (N, 139, 4)
labels = data["y"]      # (N,)

# Reduce dataset
jets = jets[:1000]
labels = labels[:1000]

print("Dataset size:", jets.shape)

# ------------------------------
# Step 2: Graph construction
# ------------------------------
def build_graph(jet, k=5):
    x = torch.tensor(jet, dtype=torch.float)
    coords = x[:, 1:3]  # eta, phi

    dist = torch.cdist(coords, coords)

    edge_index = []
    for i in range(x.shape[0]):
        knn = dist[i].topk(k+1, largest=False).indices[1:]
        for j in knn:
            edge_index.append([i, j])

    edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()

    return Data(x=x, edge_index=edge_index)

# ------------------------------
# Step 3: Convert dataset to graphs
# ------------------------------
graphs = []
for i in range(len(jets)):
    g = build_graph(jets[i])
    g.y = torch.tensor([labels[i]], dtype=torch.long)  # ensure shape [1]
    graphs.append(g)

print("Graphs built:", len(graphs))

# ------------------------------
# Step 4: Train/Validation split
# ------------------------------
train_graphs, val_graphs = train_test_split(
    graphs, test_size=0.2, random_state=42
)

train_loader = DataLoader(train_graphs, batch_size=1, shuffle=True)
val_loader = DataLoader(val_graphs, batch_size=1, shuffle=False)

# ------------------------------
# Step 5: Models
# ------------------------------
class GCN(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, hidden_channels)
        self.fc = torch.nn.Linear(hidden_channels, out_channels)

    def forward(self, data):
        x, edge_index = data.x, data.edge_index

        x = F.relu(self.conv1(x, edge_index))
        x = F.relu(self.conv2(x, edge_index))

        x = torch.mean(x, dim=0)  # global pooling (batch_size=1)
        return self.fc(x)


class GAT(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, heads=4):
        super().__init__()
        self.conv1 = GATConv(in_channels, hidden_channels, heads=heads)
        self.conv2 = GATConv(hidden_channels * heads, hidden_channels, heads=1)
        self.fc = torch.nn.Linear(hidden_channels, out_channels)

    def forward(self, data):
        x, edge_index = data.x, data.edge_index

        x = F.elu(self.conv1(x, edge_index))
        x = F.elu(self.conv2(x, edge_index))

        x = torch.mean(x, dim=0)  # global pooling
        return self.fc(x)

# ------------------------------
# Step 6: Training & Evaluation (FIXED)
# ------------------------------
def train(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0

    for data in loader:
        optimizer.zero_grad()

        out = model(data)              # [2]
        target = data.y.view(-1)       # [1]

        loss = criterion(out.view(1, -1), target)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)


def evaluate(model, loader):
    model.eval()
    correct = 0

    for data in loader:
        out = model(data)
        pred = out.argmax(dim=-1)
        correct += int(pred == data.y)

    return correct / len(loader)

# ------------------------------
# Step 7: Run Experiments
# ------------------------------
def run_experiment(model_class, name):
    model = model_class(
        in_channels=jets.shape[2],
        hidden_channels=32,
        out_channels=2
    )

    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
    criterion = torch.nn.CrossEntropyLoss()

    for epoch in range(5):
        loss = train(model, train_loader, optimizer, criterion)
        train_acc = evaluate(model, train_loader)
        val_acc = evaluate(model, val_loader)

        print(f"{name} Epoch {epoch}: "
              f"Loss {loss:.4f}, "
              f"Train Acc {train_acc:.4f}, "
              f"Val Acc {val_acc:.4f}")

# Run both models
run_experiment(GCN, "GCN")
run_experiment(GAT, "GAT")

Dataset size: (1000, 139, 4)
Graphs built: 1000
GCN Epoch 0: Loss 0.7409, Train Acc 0.7625, Val Acc 0.7500
GCN Epoch 1: Loss 0.6717, Train Acc 0.6025, Val Acc 0.5950
GCN Epoch 2: Loss 0.6352, Train Acc 0.7037, Val Acc 0.6600
GCN Epoch 3: Loss 0.6042, Train Acc 0.6625, Val Acc 0.6200
GCN Epoch 4: Loss 0.5863, Train Acc 0.7125, Val Acc 0.6900
GAT Epoch 0: Loss 0.8782, Train Acc 0.7438, Val Acc 0.6950
GAT Epoch 1: Loss 0.5807, Train Acc 0.7188, Val Acc 0.7150
GAT Epoch 2: Loss 0.5771, Train Acc 0.7438, Val Acc 0.7600
GAT Epoch 3: Loss 0.9074, Train Acc 0.7462, Val Acc 0.7300
GAT Epoch 4: Loss 0.5746, Train Acc 0.6663, Val Acc 0.6250
